# Installation Diary

### Installing Vivado in Windows



## 1.0 Cross-platform operation

Some (more) part done in Windows, some (less) part done in WSL

```
     SystemVerilog code
             │
             ▼
 WINDOWS
 ┌───────────────────────┐
 │ Vivado                │
 │ Synthesis             │
 │ Implementation        │
 │ Bitstream generation  │
 └───────────┬───────────┘
             │
       .bit file
             │
             ▼
 WSL2 / Ubuntu
 ┌───────────────────────┐
 │     openFPGALoader    │
 └───────────┬───────────┘
             │
             │ USB
             ▼
         FPGA board
```


## 1.1 Why Windows instead of Linux? 
Vivado was installed natively on Windows, while Linux-based tools such as `openFPGALoader` are run inside WSL2. 

Final arrangement: 
- Vivado → Windows
- WSL2 / Ubuntu → Linux development environment
- usbipd-win → forwards the FPGA USB device from Windows to WSL
- openFPGALoader → programs the FPGA from WSL


## 1.2 Installing Vivado 2026.1 

Vivado was installed at: 
``C:\AMDDesignTools\2026.1\Vivado`` 

The executable is: 
`C:\AMDDesignTools\2026.1\Vivado\bin\vivado.bat` 

Test: 
`powershell & "C:\AMDDesignTools\2026.1\Vivado\bin\vivado.bat" -version`

Result:
`vivado v2026.1 (64-bit)`



## 1.3 Vivado license problem

Initially Vivado reported that no valid license was available. For Vivado 2026.1 I generated the free: Vivado Basic Tier License — Node Locked from AMD's licensing page.

I selected the computer's disk serial number as the Host ID and generated the `.lic` file.

The license was then loaded using:
`Vivado License Manager → Load License → Copy License`

After installation Vivado reported:
``A valid Vivado Design Suite BASIC license has been detected.``

# 2.0 SETUP


Vivado has a GUI. But in the course, it AVOIDS using GUI, rather uses terminal to execute codes.

Follow: `https://fpga.mit.edu/6205/F26/assignments/week01/lab_setup`


-   Create proper directories in the disk: `hdl`, `sim`, `xdc`, `obj`. A build file `build.tcl` will be placed on the root directory. This file is downloadable from: `https://fpga.mit.edu/6205/_static/F26/default_files/build.tcl`. This file automates the buld process. This replaces many manual Vivado GUI/command steps with one command. The file is written according to the instructor's design of folder arrangements.
  
-  `hdl`: (hardware language) folder will contain the `.sv` (system verilog) files, that you'll write.
-  `sim`: will contain python files (running the cocotb library) intended for simulation.
-  `xdc`: will contain the constraints file for sythesis. An `.xdc` file aliases (in top level) my defined parameter (e.g., `sw0`, `led0`) for physical pin and connections (e.g., `A8`, `S5`)
-  `obj`: will contain output products, including `.bit` and information from Vivado


  ![`.bit` file creation](images\FPGA_1.jpg)
  
  
  - The final execution part is taken care by linux (WSL) rather than Windows. Then openFPGAloader sends it to the board

  ![Execution through WSL](images\FPGA_2.jpg)



# 3.0 Run a code in the FPGA

- Go to the root directory and open powershell:
  `Z:\MIT 6.205\lab01`

- Run `vivado -mode batch -source build.tcl`
This launches `vivado` in command line (:not in GUI), and executes `build.tcl`. This converts all the `.sv` files from `hdl` folder to 'bit' files at once (batch processing).  

- Once the `.bit` files are generated, you need to upload this to the FPGA board. We will use `openFPGALoader` for that. 

- But Windows has security protocol. So we need to let Windows to handover accessing the USB port to WSL. For that we use:


  -- `usbipd list` (to see available ports) (to install usbipd: `winget install --interactive --exact dorssel.usbipd-win`)

  -- `usbipd bind --busid <BUSID>` (PERMISSION: makes that Windows USB device available for sharing with WSL. `bind`= allow this USB device to be shared)

  -- `usbipd attach --wsl --busid <BUSID>` (CONNECTION anfter getting permission)

   For example, it can look like:

     -- `usbipd list` </br>
     -- `usbipd bind --busid 1-2` </br> 
     -- `usbipd attach --wsl --busid 1-2` </br>

- Go to the root folder, e.g,: `Z:\MIT 6.205\lab01` and initialize WSL.

- Command `openFPGALoader --detect` to detect if the USB port was actually shared.

- If it sees the Xilinx board, then it may show:

`index 0:` </br>
    `   idcode 0x362f093` </br>
    `   manufacturer xilinx` </br>
    `   family spartan7` </br>
    `   model xc7s50` </br>

- Then to upload the `.bit` file to the board:
   `openFPGALoader -b arty_s7_50 obj/final.bit` 

<img src="images/FPGA_3.jpg" width="300">